# MMseqs2 Homology Search

Generate Multiple Sequence Alignments (MSAs) for protein sequences by searching MMseqs2-indexed databases. GPU-by-default via [MMseqs2-GPU](https://www.nature.com/articles/s41592-025-02819-8); registry-driven dataset selection.

**Prerequisite:** the UniRef30 dataset must be provisioned. See the tool README's *Local Database Setup* section for the one-time `setup_databases.sh` invocation.

In [1]:
from proto_tools.tools.sequence_alignment.mmseqs2_homology_search import (
    Mmseqs2HomologySearchInput,
    Mmseqs2HomologySearchConfig,
    run_mmseqs2_homology_search,
)

## Single-sequence GPU search

Plain-string sugar: `queries=[seq]` becomes a singleton group with an auto-generated `sequence_id`. Defaults target UniRef30 with `use_gpu=True`.

In [2]:
ubiquitin = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"

inputs = Mmseqs2HomologySearchInput(queries=[ubiquitin])
config = Mmseqs2HomologySearchConfig()
result = run_mmseqs2_homology_search(inputs, config)

group = result.results[0]
print(f"sequence_id: {group.sequence_ids[0]}")
print(f"datasets:    {group.datasets_searched}")
print(f"homologs:    {group.num_homologs_found[0]}")

Running run_mmseqs2_homology_search [00:00]

sequence_id: seq_233b4b0b8c
datasets:    ['uniref30-2302']
homologs:    9902


## Inspect the MSA

Each group's `msas[i]` is a proto-tools `MSA` object — direct input to AlphaFold / Boltz / Chai / Protenix / AlphaFold-2.

In [3]:
msa = group.msas[0]
print(f"sequences:        {msa.num_sequences}")
print(f"alignment length: {msa.alignment_length}")
print(f"avg gap fraction: {msa.average_gap_fraction:.3f}")

sequences:        9903
alignment length: 76
avg gap fraction: 0.084


## Batch with explicit IDs

Tuples become `Mmseqs2HomologySearchQuery` objects with the supplied identifier. Phase 3 supports any number of *singleton* groups — paired (multi-chain) groups are deferred to a follow-up PR (#543).

In [4]:
batch = Mmseqs2HomologySearchInput(
    queries=[
        ("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTK", "hba_human"),
        ("MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVG", "design_001"),
    ],
)
batch_result = run_mmseqs2_homology_search(batch, Mmseqs2HomologySearchConfig())
for grp in batch_result:
    print(f"{grp.sequence_ids[0]}: {grp.num_homologs_found[0]} homologs")

Running run_mmseqs2_homology_search [00:00]

hba_human: 2778 homologs
design_001: 20 homologs


## CPU fallback

`use_gpu=False` runs the same MMseqs2 pipeline on CPU. `Config.devices_per_instance` reflects the choice (1 vs 0), so GPU schedulers and ToolPool route the call correctly.

In [5]:
cpu_config = Mmseqs2HomologySearchConfig(use_gpu=False)
print(f"GPU usage reported by config: {cpu_config.devices_per_instance}")

GPU usage reported by config: 0
